# QuestionGen UI Launcher

Primary staff-facing Colab launcher. This notebook keeps `main` as the stable default target, uses a notebook-side allowlist for pushed branches, and loads repo code from `src/` instead of reinstalling `questiongen` on routine reruns.

A clean runtime can now stay on the default settings: if required third-party packages are missing, this notebook installs them automatically before the first `questiongen` import. `BOOTSTRAP_ENV=True` remains available as a force-reinstall path. Normal rerun path: leave `BOOTSTRAP_ENV=False` and `RESET_REPO=False`. For pushed branch validation, temporarily add the branch to `REPO_BRANCH_OPTIONS`, set `REPO_BRANCH`, and use `RESET_REPO=True`. If `questiongen` was already imported in this kernel, restart the runtime before launching refreshed code.


## 1. Mount Drive And Define Standard Paths


In [ ]:
# ## 1. Mount Drive And Define Standard Paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# Edit DATA_DIR only if your runtime folder lives somewhere else in Drive.
DATA_DIR = Path("/content/drive/MyDrive/Work/Ebenezer Related/QuestionGenData")
SECRETS_DIR = DATA_DIR / "secrets"
INPUT_DIR = DATA_DIR / "input"
OUTPUT_DIR = DATA_DIR / "output"
LOGS_DIR = DATA_DIR / "logs"

for folder in [SECRETS_DIR, INPUT_DIR, OUTPUT_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("SECRETS_DIR:", SECRETS_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Minimal Settings


In [ ]:
# ## 2. Minimal Settings
API_KEY_FILE = SECRETS_DIR / "api_key.txt"
DEFAULT_DRIVE_CSV = INPUT_DIR / "questions.csv"
GRADIO_OUTPUT_DIR = OUTPUT_DIR / "gradio"

print("API_KEY_FILE:", API_KEY_FILE)
print("Default Drive CSV shown in UI:", DEFAULT_DRIVE_CSV)
print("Default Gradio output dir shown in UI:", GRADIO_OUTPUT_DIR)


## 3. Load Secrets From Drive


In [ ]:
# ## 3. Load Secrets From Drive
import os
from pathlib import Path

def load_api_keys(filepath: str | Path) -> None:
    filepath = Path(filepath)

    if not filepath.exists():
        raise FileNotFoundError(
            f"API key file not found: {filepath}\n"
            "Create it with lines like OPENAI_API_KEY=sk-..."
        )

    with filepath.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()

            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

load_api_keys(API_KEY_FILE)

MODEL_NAME = os.getenv("QUESTIONGEN_MODEL", "gpt-5-mini")
TEMPERATURE = float(os.getenv("QUESTIONGEN_TEMPERATURE", "0"))

print("Loaded env vars:")
print("- OPENAI_API_KEY:", "set" if os.getenv("OPENAI_API_KEY") else "missing")
print("- QUESTIONGEN_MODEL:", MODEL_NAME)
print("- QUESTIONGEN_TEMPERATURE:", TEMPERATURE)


## 4. Advanced Settings

`main` is the stable default branch. Add another branch to `REPO_BRANCH_OPTIONS` only when that branch is already pushed and you intentionally want Colab to validate it.

`BOOTSTRAP_ENV` is now a force-reinstall path for third-party dependencies. On a clean runtime, the notebook can auto-install missing runtime packages even when `BOOTSTRAP_ENV=False`. `RESET_REPO` deletes the existing clone and reclones the selected pushed branch. Routine reruns should usually keep both set to `False`.


In [ ]:
# ## 4. Advanced Settings
from pathlib import Path

REPO_URL = "https://github.com/AwkAwkAardvark/QuestionGen.git"
REPO_BRANCH_OPTIONS = [
    "main",
]
REPO_BRANCH = REPO_BRANCH_OPTIONS[0]
BOOTSTRAP_ENV = False
RESET_REPO = False
REPO_DIR = Path("/content/QuestionGen")

if REPO_BRANCH not in REPO_BRANCH_OPTIONS:
    raise ValueError(
        f"REPO_BRANCH must be one of {REPO_BRANCH_OPTIONS}. "
        "Update REPO_BRANCH_OPTIONS only when you intentionally want Colab to target another pushed branch."
    )

print("REPO_URL:", REPO_URL)
print("REPO_BRANCH_OPTIONS:", REPO_BRANCH_OPTIONS)
print("REPO_BRANCH:", REPO_BRANCH)
print("BOOTSTRAP_ENV:", BOOTSTRAP_ENV)
print("RESET_REPO:", RESET_REPO)
print("REPO_DIR:", REPO_DIR)


## 5. Bootstrap, Refresh, And Launch Gradio

This notebook does not run `run_batch_files(...)` directly before launch. The Gradio app is the active UI surface, and repo refresh is intentionally separated from in-kernel imports so pushed branch validation stays predictable.


In [ ]:
# ## 5. Bootstrap, Refresh, And Launch Gradio
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys

BOOTSTRAP_PACKAGES = [
    "gradio",
    "langchain-core",
    "langchain-openai",
    "pandas",
    "pydantic",
]
RUNTIME_DEPENDENCIES = (("langchain_openai", "langchain-openai"),)
UI_RUNTIME_DEPENDENCIES = RUNTIME_DEPENDENCIES + (("gradio", "gradio"),)
RESTART_REQUIRED_ENV = "QUESTIONGEN_RUNTIME_RESTART_REQUIRED"
SRC_DIR = REPO_DIR / "src"

clone_created = False
repo_updated = False

def missing_runtime_packages(dependencies):
    missing_packages = []
    for module_name, package_name in dependencies:
        if importlib.util.find_spec(module_name) is None:
            missing_packages.append(package_name)
    return missing_packages

def ensure_runtime_modules(dependencies, *, bootstrap_hint: str) -> None:
    missing_packages = missing_runtime_packages(dependencies)
    if not missing_packages:
        return
    package_list = ", ".join(f"`{package}`" for package in missing_packages)
    raise ImportError(
        f"Missing required runtime dependencies: {package_list}. Install them before continuing. {bootstrap_hint}"
    )

def mark_restart_required(reason: str) -> None:
    os.environ[RESTART_REQUIRED_ENV] = reason

def ensure_clean_questiongen_import(action_name: str) -> None:
    restart_reason = os.environ.get(RESTART_REQUIRED_ENV)
    if restart_reason:
        raise RuntimeError(f"Restart the runtime before {action_name}. {restart_reason}")
    if "questiongen" in sys.modules and (RESET_REPO or BOOTSTRAP_ENV or clone_created or repo_updated):
        mark_restart_required(
            "questiongen source was refreshed after this kernel had already imported it. Fresh-subprocess tests can still validate the updated pushed branch."
        )
        raise RuntimeError(
            "questiongen is already imported in this notebook kernel. The requested bootstrap/repo refresh step completed and the updated pushed branch source is ready, but "
            f"{action_name} requires a runtime restart for a clean import. Fresh-subprocess tests can still validate the updated pushed branch."
        )

if BOOTSTRAP_ENV and "questiongen" in sys.modules:
    raise RuntimeError(
        "questiongen is already imported in this notebook kernel. Restart the runtime before running dependency bootstrap again."
    )

setup_missing_packages = missing_runtime_packages(UI_RUNTIME_DEPENDENCIES)
auto_bootstrap = not BOOTSTRAP_ENV and bool(setup_missing_packages)

if BOOTSTRAP_ENV or auto_bootstrap:
    get_ipython().run_line_magic("pip", "install -q " + " ".join(BOOTSTRAP_PACKAGES))
    importlib.invalidate_caches()
    if auto_bootstrap:
        print("Auto-bootstrapped missing runtime dependencies:", ", ".join(setup_missing_packages))
    else:
        print("Bootstrapped third-party dependencies.")
else:
    print("Skipped third-party dependency bootstrap.")

def _git_head(repo_dir):
    completed = subprocess.run(
        ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
        check=True,
        text=True,
        capture_output=True,
    )
    return completed.stdout.strip()

previous_repo_head = None

if RESET_REPO and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
    print("Deleted existing repo clone for a clean refresh.")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    clone_created = True
    print("Cloned branch:", REPO_BRANCH)
else:
    previous_repo_head = _git_head(REPO_DIR)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    try:
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{REPO_BRANCH}"],
            check=True,
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Could not fast-forward the existing repo clone to the selected pushed branch. Rerun this setup cell with RESET_REPO=True for a clean refresh."
        ) from exc
    print("Synced existing repo clone:", REPO_DIR)

current_repo_head = _git_head(REPO_DIR)
repo_updated = previous_repo_head is not None and previous_repo_head != current_repo_head

importlib.invalidate_caches()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if importlib.util.find_spec("questiongen") is None:
    raise ModuleNotFoundError(
        "questiongen is not importable from REPO_DIR / 'src'. Check the bootstrap/repo setup output before continuing."
    )

os.environ["QUESTIONGEN_DATA_DIR"] = str(DATA_DIR)
os.environ["QUESTIONGEN_API_KEY_PATH"] = str(API_KEY_FILE)
os.environ["QUESTIONGEN_DRIVE_INPUT_CSV"] = str(DEFAULT_DRIVE_CSV)
os.environ["QUESTIONGEN_OUTPUT_DIR"] = str(GRADIO_OUTPUT_DIR)

print("Prepared branch source:", REPO_BRANCH)
print("Repo dir:", REPO_DIR)
print("Repo commit:", current_repo_head)
if clone_created:
    print("Repo state: cloned selected pushed branch into the runtime.")
elif repo_updated:
    print("Repo state: fast-forwarded existing clone to the latest pushed commit.")
else:
    print("Repo state: existing clone was already at the selected pushed commit.")
print("Source path added:", SRC_DIR)
print("Python executable:", sys.executable)

ensure_clean_questiongen_import("in-kernel Gradio launch")

ensure_runtime_modules(
    UI_RUNTIME_DEPENDENCIES,
    bootstrap_hint="If a required package is still missing after setup, rerun this setup cell with BOOTSTRAP_ENV=True. Restart the runtime if questiongen was already imported."
)

from questiongen.ui.gradio_app import create_app

app = create_app()
app.launch(debug=True)
